In [2]:
#!/usr/bin/env python3
"""
Notebook 11 — GRT p‑value vs. Lyapunov exponent for the logistic map
Gera Figure_8_lyapunov_vs_grt.tiff (600 dpi) + .png de preview
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import os

# ============================================================
# 1. Função corrigida do GRT (Barton & David, 1957)
# ============================================================
def grt_pvalue(labels):
    """Retorna R_ratio, Z_GRT, p‑value unilateral (collapse)."""
    N = len(labels)
    _, counts = np.unique(labels, return_counts=True)
    n_k = counts.astype(np.float64)

    E = 1.0 + (N**2 - np.sum(n_k**2)) / N
    R_obs = 1 + np.sum(labels[:-1] != labels[1:])
    R_ratio = R_obs / E

    p_bar = (N**2 - np.sum(n_k**2)) / (N * (N - 1))
    pi_hat = n_k / N
    q = np.sum(pi_hat * ((N - n_k) * (N - n_k - 1)) / ((N - 1) * (N - 2)))
    V_N = (N - 1) * p_bar * (1 - p_bar) + 2 * (N - 2) * (q - p_bar**2)

    Z_GRT = (R_obs - E) / np.sqrt(V_N)
    p_value = norm.cdf(Z_GRT)
    return R_ratio, Z_GRT, p_value

# ============================================================
# 2. Geração do mapa logístico e discretização simbólica
# ============================================================
def logistic_map(r, x0=0.4, N=5000):
    x = np.zeros(N)
    x[0] = x0
    for i in range(1, N):
        x[i] = r * x[i-1] * (1 - x[i-1])
    return x

def symbolize(x, partition_point=0.5):
    """Binaria a série contínua: 0 se <= partition_point, 1 caso contrário"""
    return (x >= partition_point).astype(int)

# ============================================================
# 3. Varredura de r e cálculo do expoente de Lyapunov
# ============================================================
r_values = np.linspace(3.5, 4.0, 200)
p_values = []
lyap_values = []

for r in r_values:
    # Série temporal
    x = logistic_map(r, x0=0.4, N=5000)
    # Descartar transiente
    x = x[1000:]
    # Discretização simbólica
    labels = symbolize(x, partition_point=0.5)
    # GRT p‑value
    _, _, p_val = grt_pvalue(labels)
    p_values.append(p_val)

    # Expoente de Lyapunov (aproximação simples por divergência média)
    dx = 1e-10
    x_pert = x + dx
    deriv = r * (1 - 2*x)
    lyap = np.mean(np.log(np.abs(deriv)))
    lyap_values.append(lyap)

# ============================================================
# 4. Geração da figura
# ============================================================
os.makedirs("resultados", exist_ok=True)

fig, ax = plt.subplots(figsize=(6,4))
ax.plot(lyap_values, p_values, 'o', markersize=2, alpha=0.8, color='steelblue')
ax.axhline(0.05, color='red', linestyle='--', label=r'$\alpha = 0.05$')
ax.set_xlabel(r"Lyapunov exponent $\lambda$")          # raw string usada aqui
ax.set_ylabel("GRT $p$-value")
ax.set_title("GRT $p$-value vs. Lyapunov exponent (logistic map, $r \\in [3.5, 4.0]$)")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
# Versão TIFF para a revista
plt.savefig('resultados/Figure_8_lyapunov_vs_grt.tiff', dpi=1000, format='tiff',
            pil_kwargs={'compression': 'tiff_lzw'})
# Versão PNG de preview
plt.savefig('resultados/Figure_8_lyapunov_vs_grt.png', dpi=150)
plt.close()

print("Figura 8 gerada: Figure_8_lyapunov_vs_grt.tiff (600 dpi) + preview PNG")

Figura 8 gerada: Figure_8_lyapunov_vs_grt.tiff (600 dpi) + preview PNG
